# Валютные риски: готовое видео 16:9
Запустите ячейку ниже кнопкой **▶**. Она создаст оригинальный четырёхминутный MP4 1920×1080/30 FPS и SRT, затем предложит скачать оба файла. Только бесплатные библиотеки, без озвучки: спокойная инструментальная дорожка. Цифры курсов — учебный условный пример.

In [ ]:
# Валютные риски — горизонтальный ролик 16:9, 240 с.
# Запускайте эту ячейку в Google Colab. Всё создаётся бесплатно на CPU.
import os, math, wave, subprocess, json, textwrap, shutil
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from google.colab import files

W, H, FPS, SECONDS = 1920, 1080, 30, 240
OUT = "/content/currency_risks_horizontal_240s.mp4"
MUSIC = "/content/currency_risks_calm_music.wav"
SRT = "/content/currency_risks_subtitles.srt"
assert shutil.which("ffmpeg"), "В Colab не найден ffmpeg"
FONT_BOLD = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
FONT_REG = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
if not os.path.exists(FONT_BOLD):
    raise RuntimeError("В Colab не найден DejaVu Sans. Запустите: !apt-get install fonts-dejavu-core")
F_TITLE = ImageFont.truetype(FONT_BOLD, 70)
F_TEXT = ImageFont.truetype(FONT_BOLD, 51)
F_SMALL = ImageFont.truetype(FONT_BOLD, 28)
F_ICON = ImageFont.truetype(FONT_BOLD, 118)

# Каждая строка — ровно 6 секунд: титр, фраза 0–3 с, фраза 3–6 с, предмет, персонаж.
SCENES = [
 ("КУДА ДЕЛАСЬ ПРИБЫЛЬ?", "Вчера контракт был выгодным.", "Сегодня курс изменился.", "shock", 0),
 ("СЧЁТ В ДОЛЛАРАХ", "Импортёру нужно заплатить", "десять тысяч долларов.", "invoice", 0),
 ("КУРС: 12 000", "Условный пример:", "двенадцать тысяч сумов за доллар.", "currency", 2),
 ("ПЛАН: 120 МИЛЛИОНОВ", "Так компания рассчитала", "стоимость будущей поставки.", "calculator", 0),
 ("КУРС: 13 000", "К дате платежа доллар", "стал дороже на тысячу сумов.", "chart", 2),
 ("УЖЕ 130 МИЛЛИОНОВ", "Те же десять тысяч долларов", "теперь требуют больше сумов.", "invoice", 0),
 ("РАЗНИЦА — 10 МИЛЛИОНОВ", "Вот как движение курса", "может съесть плановую прибыль.", "shock", 0),
 ("ВАЛЮТНЫЙ РИСК", "Это неопределённость результата", "из-за изменения обменного курса.", "risk", 2),
 ("КТО С НИМ СТАЛКИВАЕТСЯ?", "Импортёры, экспортёры, заёмщики", "и инвесторы в другой валюте.", "network", 1),
 ("РИСК — ЭТО НЕ ПРОГНОЗ", "Задача управления риском —", "не угадать завтрашний курс.", "chart", 2),
 ("ЗАДАЧА — УСТОЙЧИВОСТЬ", "Важно знать, сколько компания", "потеряет при плохом сценарии.", "shield", 2),
 ("ПЕРВЫЙ ТИП: ПЛАТЁЖ", "Контракт сегодня,", "оплата в иностранной валюте позже.", "invoice", 0),
 ("ТРАНЗАКЦИОННЫЙ РИСК", "Курс между этими датами", "может изменить сумму платежа.", "calendar", 0),
 ("ВТОРОЙ ТИП: БИЗНЕС", "Дорогой импорт влияет", "на цену и конкурентоспособность.", "box", 0),
 ("ЭКОНОМИЧЕСКИЙ РИСК", "Меняется не один счёт,", "а будущие доходы компании.", "chart", 1),
 ("ТРЕТИЙ ТИП: ОТЧЁТНОСТЬ", "Зарубежные активы и долги", "пересчитываются в валюту отчёта.", "papers", 2),
 ("КТО ВЫИГРЫВАЕТ?", "Ослабление местной валюты", "иногда помогает экспортной выручке.", "box", 1),
 ("НО ЕСТЬ НЮАНС", "Если у экспортёра долларовый долг,", "его расходы тоже растут.", "risk", 1),
 ("ШАГ 1: СОБРАТЬ ДАННЫЕ", "Запишите все суммы,", "валюты и даты платежей.", "papers", 2),
 ("ШАГ 2: СОПОСТАВИТЬ", "Сравните будущие поступления", "и обязательства в каждой валюте.", "balance", 2),
 ("ОТКРЫТАЯ ПОЗИЦИЯ", "Разница между ними", "показывает уязвимость к курсу.", "balance", 2),
 ("ШАГ 3: СТРЕСС-ТЕСТ", "Что будет при изменении курса", "на пять или десять процентов?", "chart", 2),
 ("ЕСТЕСТВЕННАЯ ЗАЩИТА", "Доходы и расходы", "в одной валюте частично компенсируются.", "shield", 1),
 ("ПРИМЕР: ЭКСПОРТЁР", "Получает доллары за товар", "и платит ими за сырьё.", "box", 1),
 ("ВАЛЮТНАЯ ОГОВОРКА", "Договор может заранее определить,", "как пересматривать цену.", "contract", 0),
 ("НЕ ВСЁ МОЖНО ПЕРЕЛОЖИТЬ", "Клиенты могут не принять", "весь рост цены.", "risk", 0),
 ("РЕЗЕРВ ЛИКВИДНОСТИ", "Запас денег помогает", "пережить неблагоприятный курс.", "reserve", 2),
 ("ФОРВАРД", "Стороны заранее фиксируют курс", "обмена на будущую дату.", "forward", 2),
 ("ЧТО ДАЁТ ФОРВАРД?", "Сумма платежа становится", "предсказуемее для бюджета.", "shield", 0),
 ("А ЧТО ЗАБИРАЕТ?", "Если курс стал выгоднее,", "по старому договору выгоды нет.", "balance", 0),
 ("ОПЦИОН", "Он даёт право обмена", "по согласованным условиям.", "option", 2),
 ("ЗА ПРАВО НУЖНО ПЛАТИТЬ", "Премия — цена защиты,", "даже если опцион не использован.", "coin", 2),
 ("РАЗНЫЕ ВАЛЮТЫ И СРОКИ", "Диверсификация снижает зависимость", "от одного курса и одной даты.", "network", 1),
 ("ЛИМИТЫ", "Компания ограничивает", "допустимый открытый валютный риск.", "shield", 2),
 ("СРАВНИМ ТРИ ПУТИ", "Ничего не делать,", "зафиксировать курс или купить опцион.", "balance", 2),
 ("БЕЗ ЗАЩИТЫ", "Можно выиграть на курсе,", "но можно и потерять.", "chart", 0),
 ("С ФОРВАРДОМ", "Больше определённости,", "но меньше свободы.", "forward", 0),
 ("С ОПЦИОНОМ", "Гибкость сохраняется,", "зато появляется стоимость премии.", "option", 0),
 ("ПРАВИЛО УПРАВЛЕНИЯ", "Оцените риск, цену защиты", "и способность выдержать потери.", "check", 2),
 ("ИТОГ", "Валютный риск нельзя отменить.", "Но им можно осознанно управлять.", "shield", 0),
]
assert len(SCENES) == 40
INK = (27, 43, 55)
NAVY = (25, 44, 60)
YELLOW = (246, 185, 59)
PALETTES = [(235,226,209),(219,229,222),(223,225,231),(237,223,209)]
rng = np.random.default_rng(4317)

def wrap(draw, words, font, max_w):
    out, line = [], ""
    for word in words.split():
        candidate = (line + " " + word).strip()
        if draw.textbbox((0,0), candidate, font=font)[2] > max_w and line:
            out.append(line); line = word
        else: line = candidate
    if line: out.append(line)
    return out

def make_paper(color):
    base = np.full((H, W, 3), color, dtype=np.int16)
    grain = rng.integers(-6, 7, (H, W, 1), dtype=np.int16)
    im = Image.fromarray(np.clip(base + grain, 0, 255).astype("uint8"), "RGB")
    d = ImageDraw.Draw(im)
    # Фактурные бумажные ленты и волокна, без подложки под субтитрами.
    for n in range(16):
        x = int(rng.integers(0, W)); y = int(rng.integers(0, H))
        length = int(rng.integers(28, 170))
        d.line((x,y,x+length,y-int(length*.15)), fill=(210,205,194), width=1)
    d.arc((-300,-300,900,900), 10, 250, fill=(204,198,184), width=3)
    d.arc((W-850,100,W+250,1200), 90, 330, fill=(207,201,188), width=2)
    return im
BACKGROUNDS = [make_paper(c) for c in PALETTES]

def character(kind):
    im = Image.new("RGBA", (480,660), (0,0,0,0)); d = ImageDraw.Draw(im)
    coat = [(221,150,43),(27,105,108),(133,107,172)][kind]
    skin = [(188,132,92),(177,117,84),(224,170,131)][kind]
    hair = [(36,30,32),(75,73,71),(140,68,43)][kind]
    # Белая вырубная кайма — персонаж как бумажная наклейка.
    d.rounded_rectangle((140,498,224,633), 28, fill="white")
    d.rounded_rectangle((250,498,338,633), 28, fill="white")
    d.rounded_rectangle((98,252,389,545), 70, fill="white")
    d.ellipse((153,63,334,279), fill="white")
    d.rounded_rectangle((150,504,214,622), 24, fill=(54,53,55))
    d.rounded_rectangle((260,504,327,622), 24, fill=(54,53,55))
    d.ellipse((137,605,218,645), fill=(44,43,44))
    d.ellipse((250,605,340,645), fill=(44,43,44))
    d.rounded_rectangle((111,258,374,520), 61, fill=coat)
    d.polygon([(190,265),(238,367),(270,265)], fill=(245,241,225))
    d.ellipse((165,83,318,258), fill=skin)
    if kind == 0:
        d.pieslice((145,51,340,240), 175, 350, fill=hair)
        d.ellipse((145,95,191,240), fill=hair)
    elif kind == 1:
        d.pieslice((149,62,337,195), 175, 355, fill=hair)
        d.arc((179,179,295,255),0,170,fill=hair,width=17)
        for xx in (195,260): d.ellipse((xx,155,xx+46,192), outline=(25,29,29), width=5)
        d.line((239,172,260,172),fill=(25,29,29),width=4)
    else:
        for xx, yy, rr in [(174,93,40),(218,65,44),(266,78,42),(290,125,38),(156,155,35)]:
            d.ellipse((xx-rr,yy-rr,xx+rr,yy+rr),fill=hair)
        d.ellipse((171,123,307,265),fill=skin)
    d.ellipse((198,174,207,183),fill=(33,29,31))
    d.ellipse((264,174,273,183),fill=(33,29,31))
    d.arc((219,195,259,219),0,175,fill=(94,43,40),width=3)
    d.line((125,305,91,432),fill=coat,width=42)
    d.line((356,307,410,404),fill=coat,width=42)
    d.ellipse((70,411,107,454),fill=skin)
    d.ellipse((396,389,433,430),fill=skin)
    if kind == 0:
        d.rounded_rectangle((265,345,425,455),8,fill=(247,239,218),outline=(96,84,72),width=3)
        d.line((282,373,396,373),fill=(106,94,81),width=3)
    elif kind == 1:
        d.rounded_rectangle((170,365,342,451),6,fill=(186,137,81),outline=(119,77,49),width=4)
        d.line((250,368,250,448),fill=(120,79,49),width=3)
    else:
        d.rounded_rectangle((149,354,343,450),10,fill=(101,113,127),outline=(34,44,56),width=5)
        d.rounded_rectangle((132,450,365,468),5,fill=(78,87,99))
    return im
CHARACTERS = [character(k) for k in range(3)]
shadows=[]
for s in CHARACTERS:
    sh=Image.new("RGBA",s.size,(28,34,41,0))
    sh.putalpha(s.getchannel("A").filter(ImageFilter.GaussianBlur(16)).point(lambda a:int(a*.29)))
    shadows.append(sh)

def icon(draw, kind, x, y, tm):
    x,y=int(x),int(y)
    wob=int(math.sin(tm*2.5)*10)
    y+=wob
    draw.ellipse((x-146,y-146,x+146,y+146),fill=(255,252,243),outline=(255,255,255),width=16)
    draw.ellipse((x-122,y-122,x+122,y+122),outline=NAVY,width=5)
    label={
      "shock":"?!","invoice":"$","currency":"⇄","calculator":"×",
      "chart":"↗","risk":"!","network":"↔","shield":"✓",
      "calendar":"30","box":"□","papers":"≡","balance":"=",
      "contract":"✎","reserve":"$","forward":"→","option":"?",
      "coin":"$","check":"✓"
    }.get(kind,"?")
    # DejaVu поддерживает основные знаки, без эмодзи.
    bb=draw.textbbox((0,0),label,font=F_ICON)
    draw.text((x-(bb[2]-bb[0])/2,y-(bb[3]-bb[1])/2-20),label,font=F_ICON,fill=NAVY)
    if kind in ("chart","risk","shock"):
        points=[(x-95,y+64),(x-25,y+20),(x+25,y+45),(x+90,y-55)]
        draw.line(points,fill=(226,113,80),width=12,joint="curve")
    if kind in ("shield","forward","option"):
        draw.arc((x-104,y-102,x+104,y+102),210,345,fill=YELLOW,width=13)

def draw_frame(frame):
    idx=min(39,frame//180)
    t=(frame%180)/FPS
    title,a,b,kind,who=SCENES[idx]
    im=BACKGROUNDS[idx%4].copy()
    d=ImageDraw.Draw(im)
    left_person=(idx%5==2)
    tx=720 if left_person else 112
    charx=95 if left_person else 1385
    char_y=277+int(10*math.sin(t*2.2))
    # Коллажные декоративные круги и штампы.
    d.ellipse((930,-270,1590,390),fill=(247,237,213))
    d.arc((880,-330,1660,460),15,300,fill=(200,174,139),width=3)
    for k in range(3):
        px=1170+45*k+int(5*math.sin(t*2+k))
        py=190+48*k
        d.ellipse((px,py,px+13,py+13),fill=(225,175,78))
    # Персонаж плавно покачивается вместе с тенью.
    angle=1.6*math.sin(t*1.6+idx)
    spr=CHARACTERS[who].rotate(angle,Image.Resampling.BICUBIC)
    shadow=shadows[who].rotate(angle,Image.Resampling.BICUBIC)
    im.paste(shadow,(charx+13,char_y+16),shadow)
    im.paste(spr,(charx,char_y),spr)
    d=ImageDraw.Draw(im)
    ix=1410 if left_person else 1107
    icon(d,kind,ix,535,t)
    # Крупный заголовок, отдельный цветной бумажный штамп.
    d.rounded_rectangle((tx,76,tx+430,130),radius=24,fill=NAVY)
    d.text((tx+19,85),"ВАЛЮТНЫЕ РИСКИ  /  %02d"%(idx+1),font=F_SMALL,fill=(255,255,255))
    lines=wrap(d,title,F_TITLE,970)
    intro=max(0,min(1,(t-.1)/.45))
    sx=int((1-intro)*(-180 if not left_person else 180))
    for j,line in enumerate(lines[:3]):
        d.text((tx+sx,175+j*88),line,font=F_TITLE,fill=INK,stroke_width=2,stroke_fill=(255,255,255))
    d.line((tx,470,tx+min(660,int(660*max(0,min(1,t/.7)))),470),fill=YELLOW,width=13)
    phrase=a if t<3.0 else b
    beat=t if t<3.0 else t-3.0
    appear=max(0,min(1,(beat-.04)/.34))
    # Субтитры без заднего прямоугольника.
    caption_lines=wrap(d,phrase,F_TEXT,1580)
    cap_y=919-14*(1-appear)
    for j,line in enumerate(caption_lines):
        d.text((W//2,cap_y+j*62),line,font=F_TEXT,fill=INK,anchor="mt",
               stroke_width=3,stroke_fill=(255,255,255))
    # Тонкая временная шкала для непрерывного движения.
    d.line((112,1043,1808,1043),fill=(204,197,185),width=5)
    d.line((112,1043,112+int(1696*(frame+1)/(FPS*SECONDS)),1043),fill=YELLOW,width=6)
    # Короткий мягкий бумажный переход в начале и конце каждой сцены.
    if t<.20:
        opacity=1-t/.20
    elif t>5.80:
        opacity=(t-5.80)/.20
    else: opacity=0
    if opacity:
        over=Image.new("RGB",(W,H),NAVY)
        im=Image.blend(im,over,min(.50,opacity*.50))
    return im

# Негромкая оригинальная музыка без озвучки и без платных сервисов.
def make_music():
    sr=44100
    progression=[(130.81,164.81,196.00),(110.00,130.81,164.81),
                 (87.31,130.81,174.61),(98.00,146.83,196.00)]
    with wave.open(MUSIC,"wb") as wf:
        wf.setnchannels(2); wf.setsampwidth(2); wf.setframerate(sr)
        for second in range(SECONDS):
            t=second+np.arange(sr)/sr
            chord=progression[int(second//10)%4]
            sig=np.zeros(sr,dtype=np.float64)
            for j,freq in enumerate(chord):
                sig+=(0.11/(j+1))*np.sin(2*np.pi*freq*t)
            pulse=((t*100/60)%1)
            sig+=0.028*np.exp(-pulse*7)*np.sin(2*np.pi*440*t)
            sig+=0.012*np.exp(-pulse*13)*np.sin(2*np.pi*65*t)
            fade=np.minimum(1,np.minimum(t/2.5,(SECONDS-t)/3.0))
            sig=np.clip(sig*fade,-0.8,0.8)
            lr=np.stack((sig,sig*.93),axis=1)
            wf.writeframes((lr*32767).astype("<i2").tobytes())

def ts(seconds):
    millis=round(seconds*1000)
    h=millis//3600000; millis%=3600000
    m=millis//60000; millis%=60000
    s=millis//1000; millis%=1000
    return f"{h:02}:{m:02}:{s:02},{millis:03}"
with open(SRT,"w",encoding="utf-8") as sub:
    for idx,(_,a,b,_,_) in enumerate(SCENES):
        for half,phrase in enumerate((a,b)):
            start=idx*6+half*3
            sub.write(f"{idx*2+half+1}\n{ts(start)} --> {ts(start+3)}\n{phrase}\n\n")

print("Создаю музыку...")
make_music()
cmd=["ffmpeg","-y","-hide_banner","-loglevel","error",
     "-f","rawvideo","-vcodec","rawvideo","-pix_fmt","rgb24",
     "-s",f"{W}x{H}","-r",str(FPS),"-i","pipe:0","-i",MUSIC,
     "-c:v","libx264","-preset","veryfast","-crf","23",
     "-pix_fmt","yuv420p","-c:a","aac","-b:a","128k",
     "-frames:v",str(FPS*SECONDS),"-movflags","+faststart","-shortest",OUT]
print("Рендер 40 сцен. На бесплатном CPU это может занять некоторое время.")
proc=subprocess.Popen(cmd,stdin=subprocess.PIPE)
try:
    for n in range(FPS*SECONDS):
        proc.stdin.write(draw_frame(n).tobytes())
        if n%180==0:
            print(f"Сцена {n//180+1}/40",flush=True)
finally:
    proc.stdin.close()
return_code=proc.wait()
if return_code: raise RuntimeError(f"ffmpeg завершился с кодом {return_code}")
probe=json.loads(subprocess.check_output(["ffprobe","-v","error","-show_entries",
    "format=duration:stream=width,height,r_frame_rate","-of","json",OUT]))
duration=float(probe["format"]["duration"])
assert abs(duration-240)<0.15, duration
assert any(s.get("width")==1920 and s.get("height")==1080 and s.get("r_frame_rate")=="30/1" for s in probe["streams"])
print(f"Готово: {duration:.2f} с; {os.path.getsize(OUT)/1e6:.1f} МБ.")
files.download(OUT)
files.download(SRT)
